# Video 04 - Exercise Solutions

Try the exercises in [exercises.md](../exercises.md) yourself first. These are **one** good answer each.

In [0]:
import json
from databricks.sdk import WorkspaceClient

client = WorkspaceClient().serving_endpoints.get_open_ai_client()
MODEL = "databricks-meta-llama-3-3-70b-instruct"

class Chat:
    def __init__(self, system="You are a friendly, concise assistant. Keep answers under 40 words.", max_history=None):
        self.system = {"role": "system", "content": system}
        self.history = []
        self.max_history = max_history
        self.total_input_tokens = 0      # Exercise 2: running total

    def send(self, text):
        self.history.append({"role": "user", "content": text})
        recent = self.history if self.max_history is None else self.history[-self.max_history:]
        response = client.chat.completions.create(model=MODEL, messages=[self.system] + recent)
        reply = response.choices[0].message.content
        self.history.append({"role": "assistant", "content": reply})
        self.total_input_tokens += response.usage.prompt_tokens
        return reply

In [0]:
chat = Chat()
chat.send("My favourite colour is blue.")
chat.send("Suggest a T-shirt design for me.")

# Rewrite history: the model will believe whatever is in the list
chat.history[0]["content"] = "My favourite colour is green."
print(chat.send("What is my favourite colour?"))

# Lesson: YOU control the model's memory completely. That's powerful - and a
# reason to protect stored histories, since editing them changes what the model 'knows'.

In [0]:
turns = [
    "Hi, I'm Sam from Perth.", "Suggest a weekend activity.", "What should I pack?",
    "Where can I eat nearby?", "How long will it take?", "What if it rains?",
    "Suggest a backup plan.", "What should I wear?", "Any photo spots?", "Summarise my plan.",
]

full, trimmed = Chat(), Chat(max_history=4)
for t in turns:
    full.send(t)
    trimmed.send(t)

print(f"Full history total input tokens : {full.total_input_tokens}")
print(f"Trimmed (last 4) total          : {trimmed.total_input_tokens}")
print(f"Full history costs {full.total_input_tokens / trimmed.total_input_tokens:.1f}x more input")

In [0]:
import tempfile, os

path = os.path.join(tempfile.gettempdir(), "chat_history.json")
with open(path, "w") as f:
    json.dump(full.history, f)          # save the memory

resumed = Chat()
with open(path) as f:
    resumed.history = json.load(f)      # load it into a brand-new chat

print(resumed.send("Remind me: what is my name and what was my plan?"))

# For 1,000 users you'd store one history per user (e.g. in a table keyed by user id),
# load it at the start of each request and save it at the end.

In [0]:
old_messages = full.history[:-4]   # everything except the last 4
old_text = "\n".join(f"{m['role']}: {m['content']}" for m in old_messages)

summary = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": f"Summarise this conversation in 2 sentences. Keep names and places.\n\n{old_text}"}],
).choices[0].message.content
print("Summary:", summary, "\n")

summarised = Chat(system=f"You are a friendly, concise assistant.\nEarlier conversation summary: {summary}", max_history=4)
summarised.history = full.history[-4:]
print(summarised.send("What is my name and where do I live?"))

# A summary keeps the gist but may drop details you later need; a pinned fact never will.